# Модель SIR (Восприимчивые - Инфицированные - Выздоровевшие)

**Цель работы:** Исследовать динамику эпидемического процесса с помощью
компартментальной модели SIR.

## Теоретическое введение

Модель SIR описывается системой дифференциальных уравнений:

$$
\begin{cases}
\frac{dS}{dt} = -\beta \frac{IS}{N} \\
\frac{dI}{dt} = \beta \frac{IS}{N} - \gamma I \\
\frac{dR}{dt} = \gamma I
\end{cases}
$$

In [ ]:
using DrWatson
@quickactivate "sir_lv_project"

using DifferentialEquations
using DataFrames
using Plots
using LaTeXStrings
using JLD2

Создание директорий для результатов

In [ ]:
script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(plotsdir(script_name))
mkpath(datadir(script_name))

## Определение модели
Функция, описывающая правые части системы

In [ ]:
function sir_ode!(du, u, p, t)
    S, I, R = u
    β, γ = p
    N = S + I + R

    du[1] = -β * I * S / N
    du[2] = β * I * S / N - γ * I
    du[3] = γ * I
    nothing
end

## Параметры модели
Зададим параметры для базового сценария

In [ ]:
β = 0.4      # коэффициент заражения
γ = 0.1      # коэффициент выздоровления
p = [β, γ]

Начальные условия

In [ ]:
N = 1000.0   # общая численность популяции
I0 = 5.0     # начальное число инфицированных
R0 = 0.0     # начальное число выздоровевших
S0 = N - I0 - R0
u0 = [S0, I0, R0]

Временной интервал

In [ ]:
tspan = (0.0, 160.0)

Расчет базового репродуктивного числа

In [ ]:
R0_value = β / γ
println("Базовое репродуктивное число R0 = ", round(R0_value, digits=2))

## Решение системы ОДУ

In [ ]:
prob = ODEProblem(sir_ode!, u0, tspan, p)
sol = solve(prob, Tsit5(), saveat=1.0)

Создаем DataFrame с результатами

In [ ]:
df = DataFrame(t = sol.t)
df.S = [u[1] for u in sol.u]
df.I = [u[2] for u in sol.u]
df.R = [u[3] for u in sol.u]

## Визуализация результатов
### График динамики всех трех групп

In [ ]:
p1 = plot(df.t, [df.S df.I df.R],
          label=[L"S(t)" L"I(t)" L"R(t)"],
          xlabel="Время (дни)",
          ylabel="Численность",
          title="Модель SIR: Динамика эпидемии",
          linewidth=2,
          palette=[:blue :red :green],
          grid=true,
          size=(800, 500))

Сохраняем график

In [ ]:
savefig(p1, plotsdir(script_name, "sir_dynamics.png"))

### График только инфицированных

In [ ]:
p2 = plot(df.t, df.I,
          label=L"I(t)",
          xlabel="Время (дни)",
          ylabel="Количество инфицированных",
          title="Динамика числа зараженных",
          color=:red,
          linewidth=2,
          grid=true,
          size=(800, 400))

savefig(p2, plotsdir(script_name, "sir_infected.png"))

## Анализ результатов

In [ ]:
peak_idx = argmax(df.I)
peak_time = df.t[peak_idx]
peak_value = df.I[peak_idx]

println("\nАнализ результатов:")
println("  Пик эпидемии: I_max = ", round(peak_value, digits=1),
        " на ", round(peak_time, digits=1), " день")
println("  Переболело всего: R(∞) = ", round(df.R[end], digits=1))

## Сохранение результатов

In [ ]:
@save datadir(script_name, "sir_results.jld2") df p R0_value
println("\nРезультаты сохранены в data/$(script_name)/")